In [31]:
import pandas as pd
import numpy as np
import os
import warnings
import geopandas as gpd
import plotly.graph_objects as go
import plotly.express as px
from sodapy import Socrata

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

DATA_DIR = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"
os.chdir(DATA_DIR)
print("Directorio de trabajo:", os.getcwd())

Directorio de trabajo: C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data


**Precipitación historica**

In [ ]:
# ── Carga ambos archivos CSV históricos con columnas mínimas ─────────────────
STATION_COLS = [
    "CodigoEstacion", "NombreEstacion", "Departamento",
    "Municipio", "ZonaHidrografica", "Latitud", "Longitud",
]
LOAD_COLS = STATION_COLS + ["FechaObservacion", "ValorObservado"]


def _load_precip(path, decimal="."):
    chunks = []
    for chunk in pd.read_csv(
        path,
        usecols=LOAD_COLS,
        decimal=decimal,
        low_memory=False,
        chunksize=200_000,   # ~200k rows at a time
    ):
        # Normalize numeric columns
        for col in ("Latitud", "Longitud", "ValorObservado"):
            if chunk[col].dtype == object:
                chunk[col] = pd.to_numeric(
                    chunk[col].str.replace(",", ".", regex=False).str.strip(),
                    errors="coerce",
                )
        chunk["fecha"] = pd.to_datetime(
            chunk["FechaObservacion"].str[:11],
            format="%Y %b %d", errors="coerce", cache=True,
        )
        chunks.append(chunk.drop(columns="FechaObservacion"))
    return pd.concat(chunks, ignore_index=True)

df1 = _load_precip("Precipitación_20251222.csv")
df2 = _load_precip("Precipitación_20251222_2.csv", decimal=",")

precip_raw = pd.concat([df1, df2], ignore_index=True)
del df1, df2  # libera memoria

print(f"Registros cargados: {len(precip_raw):,}")

ParserError: Error tokenizing data. C error: out of memory

In [ ]:
# ── Agrega a nivel diario ─────────────────────────────────────────────────────
df_daily = (
    precip_raw
    .dropna(subset=["ValorObservado", "fecha"])
    .drop_duplicates(subset=STATION_COLS + ["fecha"])          # evita duplicados entre archivos
    .groupby(STATION_COLS + ["fecha"], as_index=False, sort=False)
    .agg(precip_acum_diaria=("ValorObservado", "sum"))
)
del precip_raw  # libera ~1 GB

df_daily["mes"] = df_daily["fecha"].dt.month
print(f"Registros diarios: {len(df_daily):,} | Estaciones: {df_daily['CodigoEstacion'].nunique():,}")

# ── Z-score mensual por estación → bandera de alerta ──────────────────────────
stats = (
    df_daily
    .groupby(["CodigoEstacion", "mes"])["precip_acum_diaria"]
    .agg(hist_mean="mean", hist_std="std")
    .reset_index()
)
df_daily = df_daily.merge(stats, on=["CodigoEstacion", "mes"], how="left")
df_daily["alerta_2sigma"] = (
    (df_daily["precip_acum_diaria"] - df_daily["hist_mean"]) / df_daily["hist_std"]
) > 2

# Umbral: percentil 85 de la proporción de días en alerta por estación (top 15%)
UMBRAL = df_daily.groupby("CodigoEstacion")["alerta_2sigma"].mean().quantile(0.85)
print(f"Umbral top 15% (prop. días en alerta): {UMBRAL:.4f}")

# ── Tabla resumen por estación (coordenadas + flag) ───────────────────────────
estaciones_alerta = (
    df_daily
    .groupby(STATION_COLS, as_index=False)
    .agg(prop_alertas=("alerta_2sigma", "mean"))
)
estaciones_alerta["alerta"]   = (estaciones_alerta["prop_alertas"] >= UMBRAL).astype(int)
estaciones_alerta["cod_norm"] = estaciones_alerta["CodigoEstacion"].astype(str).str.lstrip("0")

print(f"Estaciones con alerta histórica: {estaciones_alerta['alerta'].sum()} / {len(estaciones_alerta)}")
estaciones_alerta.head()

In [ ]:
# ── Carga davipola y proyecta a EPSG 3116 ────────────────────────────────────
# (se hace una sola vez sobre ~1400 estaciones únicas, no sobre los 978K registros diarios)
MENSUALES_DIR = (
    r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos"
    r"\2025\Outputs\Output1\Stress Test\3.Data"
    r"\Escenarios Cambio Climatico IDEAM IV comunicacion\Mensuales"
)
davipola = pd.read_excel(os.path.join(MENSUALES_DIR, "davipola_dane.xlsx"))

gdf_mun = gpd.GeoDataFrame(
    davipola,
    geometry=gpd.points_from_xy(davipola.LONGITUD, davipola.LATITUD),
    crs="EPSG:4326",
).to_crs(epsg=3116)

print(f"Municipios cargados: {len(gdf_mun)}")

In [ ]:
df_daily_vf.shape

(978145, 21)

**Directamente desde la API**

In [ ]:
# ── Consulta API IDEAM – último día disponible ────────────────────────────────
DATASET_ID = "s54a-sgyg"
client = Socrata("www.datos.gov.co", None)

fecha_mapa = client.get(DATASET_ID, select="max(fechaobservacion)")[0]["max_fechaobservacion"][:10]
print(f"Última fecha disponible: {fecha_mapa}")

where = (
    f"fechaobservacion >= '{fecha_mapa}T00:00:00' "
    f"AND fechaobservacion < '{fecha_mapa}T23:59:59.999'"
)
records, offset = [], 0
while True:
    batch = client.get(DATASET_ID, where=where, limit=100_000, offset=offset)
    if not batch:
        break
    records.extend(batch)
    offset += 100_000
    print(f"  {len(records):,} registros descargados…")
client.close()

# ── Agrega a nivel de estación ────────────────────────────────────────────────
df_api = pd.DataFrame.from_records(records)
for col in ("valorobservado", "latitud", "longitud"):
    df_api[col] = pd.to_numeric(df_api[col], errors="coerce")
df_api = df_api.dropna(subset=["latitud", "longitud", "valorobservado"])
df_api = df_api[df_api["valorobservado"] >= 0]

STATION_COLS_API = [
    "codigoestacion", "nombreestacion", "departamento",
    "municipio", "zonahidrografica", "latitud", "longitud",
]
df_dia = (
    df_api.groupby(STATION_COLS_API, as_index=False)
    .agg(
        precip_acum_mm   = ("valorobservado", "sum"),
        precip_max_10min = ("valorobservado", "max"),
        n_lecturas       = ("valorobservado", "count"),
    )
)
del df_api

# ── Fusión con alerta histórica por código de estación ───────────────────────
df_dia["cod_norm"]    = df_dia["codigoestacion"].str.lstrip("0")
df_dia = df_dia.merge(
    estaciones_alerta[["cod_norm", "alerta", "prop_alertas"]],
    on="cod_norm", how="left"
)
df_dia["alerta"]       = df_dia["alerta"].fillna(0).astype(int)
df_dia["prop_alertas"] = df_dia["prop_alertas"].fillna(0)

# ── Sjoin sobre ~700 estaciones → asignación de municipio ────────────────────
# Mucho más rápido que el sjoin original sobre 978K registros diarios
gdf_dia = gpd.GeoDataFrame(
    df_dia,
    geometry=gpd.points_from_xy(df_dia.longitud, df_dia.latitud),
    crs="EPSG:4326",
).to_crs(epsg=3116)

# how="left": todos los municipios de davipola reciben la estación más cercana
df_muni = gpd.sjoin_nearest(gdf_mun, gdf_dia, how="left", distance_col="dist_m")
df_muni = (
    df_muni
    .groupby(["COD_MPIO", "NOM_MPIO", "NOM_DPTO", "LATITUD", "LONGITUD"], as_index=False)
    .agg(
        precip_acum_mm   = ("precip_acum_mm",  "mean"),  # promedio de estaciones asignadas
        precip_max_10min = ("precip_max_10min", "max"),
        n_estaciones     = ("codigoestacion",   "count"),
        alerta           = ("alerta",           "max"),  # 1 si alguna estación tiene alerta
        prop_alertas     = ("prop_alertas",     "max"),
    )
)
df_muni["alerta"] = df_muni["alerta"].fillna(0).astype(int)

print(f"\nEstaciones activas hoy  : {len(df_dia):,}")
print(f"  Con alerta histórica  : {df_dia['alerta'].sum():,}")
print(f"Municipios con datos    : {len(df_muni):,}")

In [ ]:
# ── Categorías y paleta ───────────────────────────────────────────────────────
ORDEN_CAT = [
    "Sin lluvia (0 mm)", "Muy ligera (< 5 mm)", "Ligera (5–15 mm)",
    "Moderada (15–30 mm)", "Fuerte (30–60 mm)", "Muy fuerte (60–100 mm)", "Extrema (≥ 100 mm)",
]
COLOR_MAP = {
    "Sin lluvia (0 mm)":      "#d4e6f1",
    "Muy ligera (< 5 mm)":    "#85c1e9",
    "Ligera (5–15 mm)":       "#2e86c1",
    "Moderada (15–30 mm)":    "#1a5276",
    "Fuerte (30–60 mm)":      "#f39c12",
    "Muy fuerte (60–100 mm)": "#e74c3c",
    "Extrema (≥ 100 mm)":     "#7b241c",
}

def categoria_precip(mm):
    if mm == 0:    return "Sin lluvia (0 mm)"
    elif mm < 5:   return "Muy ligera (< 5 mm)"
    elif mm < 15:  return "Ligera (5–15 mm)"
    elif mm < 30:  return "Moderada (15–30 mm)"
    elif mm < 60:  return "Fuerte (30–60 mm)"
    elif mm < 100: return "Muy fuerte (60–100 mm)"
    else:          return "Extrema (≥ 100 mm)"

df_muni["categoria"] = pd.Categorical(
    df_muni["precip_acum_mm"].apply(categoria_precip),
    categories=ORDEN_CAT, ordered=True,
)
df_muni["tamaño"] = (df_muni["precip_acum_mm"].clip(lower=1) ** 0.45 * 3).clip(lower=5, upper=22)

# ── Mapa combinado (coordenadas de municipio desde davipola) ─────────────────
fig = go.Figure()

# Capa 1: precipitación actual por categoría de intensidad (un punto por municipio)
for cat in ORDEN_CAT:
    sub = df_muni[df_muni["categoria"] == cat]
    if sub.empty:
        continue
    fig.add_trace(go.Scattermapbox(
        lat=sub["LATITUD"],
        lon=sub["LONGITUD"],
        mode="markers",
        marker=dict(size=sub["tamaño"], color=COLOR_MAP[cat], opacity=0.85),
        name=cat,
        text=sub["NOM_MPIO"],
        customdata=sub[[
            "NOM_DPTO", "precip_acum_mm", "precip_max_10min",
            "n_estaciones", "alerta",
        ]].values,
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Departamento: %{customdata[0]}<br>"
            "Precipitación acumulada: %{customdata[1]:.1f} mm<br>"
            "Máx. en 10 min: %{customdata[2]:.1f} mm<br>"
            "Estaciones asignadas: %{customdata[3]}<br>"
            "Alerta histórica: %{customdata[4]}<br>"
            "<extra></extra>"
        ),
    ))

# Capa 2: anillo naranja en municipios con alerta histórica
alert_muni = df_muni[df_muni["alerta"] == 1]
if not alert_muni.empty:
    fig.add_trace(go.Scattermapbox(
        lat=alert_muni["LATITUD"],
        lon=alert_muni["LONGITUD"],
        mode="markers",
        marker=dict(size=alert_muni["tamaño"] + 7, color="rgba(255,140,0,0.45)"),
        name="⚠️ Alerta histórica (top 15%)",
        text=alert_muni["NOM_MPIO"],
        customdata=alert_muni[["NOM_DPTO", "precip_acum_mm", "prop_alertas"]].values,
        hovertemplate=(
            "<b>%{text}</b> ⚠️<br>"
            "Departamento: %{customdata[0]}<br>"
            "Precipitación hoy: %{customdata[1]:.1f} mm<br>"
            "Tasa alertas históricas: %{customdata[2]:.1%}<br>"
            "<extra></extra>"
        ),
    ))

fig.update_layout(
    mapbox_style="carto-positron",
    mapbox_zoom=5,
    mapbox_center={"lat": 4.5, "lon": -74.0},
    height=800,
    title=dict(
        text=(
            f"<b>Precipitación diaria y alerta histórica – Colombia</b><br>"
            f"<sup>Fecha: {fecha_mapa} | Fuente: IDEAM – datos.gov.co | "
            f"Anillo naranja = top 15% alertas históricas</sup>"
        ),
        x=0.5,
    ),
    legend_title_text="Intensidad / Alerta",
    margin={"r": 0, "t": 70, "l": 0, "b": 0},
)
fig.show()